In [2]:
import os
os.environ.setdefault('http_proxy', 'http://webproxy.au.harveynorman.com:8080')
os.environ.setdefault('https_proxy', 'http://webproxy.au.harveynorman.com:8080')
os.environ.setdefault('AWS_DEFAULT_REGION', 'ap-southeast-2')

'ap-southeast-2'

## Setup data in S3

In [9]:
!aws s3 ls s3://romadv-itv-retail/retail_db/ --profile itvadmin --recursive

In [10]:
!ls ~/itversity/Research/data/retail_db

README.md      create_db_tables_pg.sql	load_db_tables_pg.sql  products
categories     customers		order_items
create_db.sql  departments		orders


In [11]:
!aws s3 cp ~/itversity/Research/data/retail_db s3://romadv-itv-retail/retail_db --recursive --profile itvadmin

upload: ../../itversity/Research/data/retail_db/create_db_tables_pg.sql to s3://romadv-itv-retail/retail_db/create_db_tables_pg.sql
upload: ../../itversity/Research/data/retail_db/departments/part-00000 to s3://romadv-itv-retail/retail_db/departments/part-00000
upload: ../../itversity/Research/data/retail_db/README.md to s3://romadv-itv-retail/retail_db/README.md
upload: ../../itversity/Research/data/retail_db/categories/part-00000 to s3://romadv-itv-retail/retail_db/categories/part-00000
upload: ../../itversity/Research/data/retail_db/products/part-00000 to s3://romadv-itv-retail/retail_db/products/part-00000
upload: ../../itversity/Research/data/retail_db/customers/part-00000 to s3://romadv-itv-retail/retail_db/customers/part-00000
upload: ../../itversity/Research/data/retail_db/create_db.sql to s3://romadv-itv-retail/retail_db/create_db.sql
upload: ../../itversity/Research/data/retail_db/orders/part-00000 to s3://romadv-itv-retail/retail_db/orders/part-00000
upload: ../../itversity/

In [12]:
!aws s3 ls s3://romadv-itv-retail/retail_db/ --profile itvadmin --recursive

2026-01-09 17:04:59        806 retail_db/README.md
2026-01-09 17:04:59       1029 retail_db/categories/part-00000
2026-01-09 17:04:59   10303297 retail_db/create_db.sql
2026-01-09 17:04:59       1748 retail_db/create_db_tables_pg.sql
2026-01-09 17:05:00     953719 retail_db/customers/part-00000
2026-01-09 17:04:59         60 retail_db/departments/part-00000
2026-01-09 17:04:59   10297372 retail_db/load_db_tables_pg.sql
2026-01-09 17:05:01    5408880 retail_db/order_items/part-00000
2026-01-09 17:05:00    2999944 retail_db/orders/part-00000
2026-01-09 17:04:59     174155 retail_db/products/part-00000


## Copy Database and Tables for Redshift COPY command

```sql
-- Create Database and Table for Redshift Copy Command
CREATE DATABASE retail_db;

DROP TABLE IF EXISTS orders;
CREATE TABLE orders (
  order_id INT,
  order_date DATE,
  order_customer_id INT,
  order_status VARCHAR(30)
);
```

### Create IAM user with programmatic access having required privileges on s3 (full access) in the AWS Console

Make user user has S3 Full Access policy and save the credentials of the user to be used for later.

## Run Copy CMD to copy data from s3 to redshift tables

In [18]:
# the files in s3 below will be copied to a redshift table
!aws s3 ls s3://romadv-itv-retail/retail_db/orders/ --recursive --profile itvgithub

2026-01-09 17:05:00    2999944 retail_db/orders/part-00000


```sql
-- COPY cmd
copy orders from 's3://romadv-itv-retail/retail_db/orders/part-00000' 
credentials 'aws_access_key_id=<your access key>;aws_secret_access_key=<your secret key>'
csv;
```
will result with the error below initially:

ERROR: Load into table 'orders' failed. Check 'sys_load_error_detail' system table for details.

To find the error use:
```sql
-- for redshift cluster
SELECT * FROM stl_load_errors

--for redshift serverless
SELECT * FROM SYS_LOAD_ERROR_DETAIL;
```
Error Message: Invalid Date Format - length must be 10 or more

The error is due to the incorrect format used in the order_date column (date type) but the format in the source column is datetime type.

Solution: Recreate the table with the appropriate column type
```sql
DROP TABLE IF EXISTS orders;
CREATE TABLE orders (
  order_id INT PRIMARY KEY,
  order_date DATETIME,
  order_customer_id INT,
  order_status VARCHAR(30)
);
```

Rerun the COPY cmd
```sql
COPY orders FROM 's3://romadv-itv-retail/retail_db/orders/part-00000' 
CREDENTIALS 'aws_access_key_id=<your access key>;aws_secret_access_key=<your secret key>'
CSV;

-- validation
SELECT * FROM orders LIMIT 10;

SELECT count(1) FROM orders;

SELECT order_status, COUNT(*) AS order_count
FROM orders
GROUP BY order_status;
```
Copy Command Docs: https://docs.aws.amazon.com/redshift/latest/dg/r_COPY.html

## Creating IAM Role for Redshift access to S3

In [22]:
# the files in s3 below will be copied to a redshift table
!aws s3 ls s3://romadv-itv-retail/retail_db/order_items/ --recursive --profile itvgithub

2026-01-09 17:05:01    5408880 retail_db/order_items/part-00000


### Create IAM Role with s3 full access

* Create ITVRedshiftGithubS3FullAccessRole role for Redshift Service with s3 full access policy (ITVGitHubS3FullPolicy).
* Add role to Redshift cluster (Actions -> Manage Permissions -> Manage IAM).
* Wait until the server is modified and in available state.

```sql
-- Copy data from s3 using IAM Role
DROP TABLE IF EXISTS order_items;
CREATE TABLE order_items (
  order_item_id INT PRIMARY KEY,
  order_item_order_id INT,
  order_item_product_id INT,
  order_item_quantity INT,
  order_item_subtotal FLOAT,
  order_item_product_price FLOAT
);

COPY order_items FROM 's3://romadv-itv-retail/retail_db/order_items/part-00000' 
IAM_ROLE 'arn:aws:iam::222634372385:role/ITVRedshiftGithubS3FullAccessRole'
DELIMITER ',';

SELECT * FROM order_items LIMIT 10;

SELECT count(*) FROM order_items;
```

### Supported file formats of Redshift Copy Command

In [23]:
# the files in s3 below will be copied to a redshift table
!aws s3 ls s3://romadv-itv-retail/retail_db_json/order_items/ --recursive --profile itvgithub

2025-12-29 11:12:41   28655610 retail_db_json/order_items/part-r-00000-6b83977e-3f20-404b-9b5f-29376ab1419e


```sql
-- Copy JSON Files
DROP TABLE IF EXISTS order_items;

CREATE TABLE order_items (
  order_item_id INT PRIMARY KEY,
  order_item_order_id INT,
  order_item_product_id INT,
  order_item_quantity INT,
  order_item_subtotal FLOAT,
  order_item_product_price FLOAT
);

COPY order_items FROM 's3://romadv-itv-retail/retail_db_json/order_items'
IAM_ROLE 'arn:aws:iam::222634372385:role/ITVRedshiftGithubS3FullAccessRole'
JSON AS 'auto';

SELECT * FROM order_items LIMIT 10;

SELECT count(*) FROM order_items;
```